# Stage 3 — RL Training: GRPO + PPO
### Space Flight AI

**Phase A — GRPO Reasoning Training (no KSP needed, runs on H200)**
- Generates 8 responses per prompt, scores them, learns from best
- Reward = format + reasoning depth + decision quality
- Expected: 2-4 hours

**Phase B — PPO Flight Control (requires KSP + SSH tunnel)**
- Curriculum: Level 1 hover → Level 2 VTOL → Level 3 orbit → ...
- Advances after 3 successes AND avg reward > 80% threshold
- Action space: throttle + pitch + heading (Phase 1)
- Every flight log saved automatically
- Space engineer reviews logs after Level 3 works

**Prerequisites:** `space_flight_ai_model/` from notebook 02

## Step 0 — Install Dependencies

In [ ]:
!pip install trl transformers peft accelerate bitsandbytes torch datasets wandb krpc -q

## Step 1 — Imports and Config

In [ ]:
import os, json, time, re, random, math, torch, wandb
from pathlib import Path
from datetime import datetime
from collections import deque
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    LogitsProcessor, LogitsProcessorList,
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import GRPOTrainer, GRPOConfig

# Load environment
with open("/home/jovyan/spacemechanics_fine_tune_rl/.env") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()

BASE_MODEL         = "Qwen/Qwen2.5-7B-Instruct"
SFT_MODEL_DIR      = "./space_flight_ai_model"
GRPO_OUTPUT_DIR    = "./grpo_checkpoints"
PPO_OUTPUT_DIR     = "./ppo_checkpoints"
FINAL_RL_MODEL_DIR = "./space_flight_ai_rl_model"
FLIGHT_LOG_DIR     = "./flight_logs"
TRAINING_DATA      = "./finetune_ready.jsonl"
HF_TOKEN           = os.environ.get("HF_TOKEN")
WANDB_KEY          = os.environ.get("WANDB_API_KEY")

# GRPO
GRPO_EPOCHS      = 2
GRPO_BATCH_SIZE  = 4
GRPO_LR          = 1e-5
GRPO_GROUP_SIZE  = 8

# KSP via SSH tunnel
# Local machine runs: ssh -R 50000:localhost:50000 -R 50001:localhost:50001 jovyan@<H200-IP>
KSP_HOST         = "127.0.0.1"
KSP_RPC_PORT     = 50000
KSP_STREAM_PORT  = 50001

# Curriculum advancement
SUCCESSES_TO_ADVANCE = 3
REWARD_THRESHOLD_PCT = 0.80

for d in [GRPO_OUTPUT_DIR, PPO_OUTPUT_DIR, FLIGHT_LOG_DIR, FINAL_RL_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"PyTorch          : {torch.__version__}")
print(f"CUDA             : {torch.cuda.is_available()}")
print(f"GPU              : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"SFT model exists : {Path(SFT_MODEL_DIR).exists()}")

---
# PHASE A — GRPO Reasoning Training
### No KSP needed. Runs entirely on H200.
---

## Step 2 — Load Fine-Tuned Model

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ HuggingFace login OK")

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"✓ Tokenizer loaded — vocab: {len(tokenizer)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True, token=HF_TOKEN,
)
base.resize_token_embeddings(len(tokenizer))
base.config.vocab_size = len(tokenizer)
base.generation_config.vocab_size = len(tokenizer)
base.config.use_cache = False

model = PeftModel.from_pretrained(base, SFT_MODEL_DIR)
model = prepare_model_for_kbit_training(model)

print(f"✓ SFT model loaded")
print(f"  Vocab match : {base.config.vocab_size == len(tokenizer)}")
if torch.cuda.is_available():
    print(f"  VRAM        : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## Step 3 — GRPO Reward Functions

Rule-based rewards matching DeepSeek R1 paper:
`Reward_rule = Reward_format + Reward_accuracy`

In [ ]:
REQUIRED_TOKENS = ["<|reasoning|>", "<|decision|>", "<|theory_ref|>"]
PHYSICS_KW = [
    "delta-v","orbit","velocity","altitude","thrust","burn","trajectory",
    "periapsis","apoapsis","gravity","atmosphere","fuel","maneuver",
    "guidance","pitch","prograde","retrograde","hohmann","insertion",
    "circularization","dynamic pressure","staging","max-q","throttle",
]
ACTION_KW = [
    "throttle","pitch","burn","stage","reduce","increase","maintain",
    "adjust","activate","cut","ignite","circularize","deorbit","dock",
    "separate","prograde","retrograde",
]

def format_reward(responses, **kw):
    return [sum(1.0 for t in REQUIRED_TOKENS if t in r) for r in responses]

def reasoning_reward(responses, **kw):
    out = []
    for r in responses:
        s = 0.0
        txt = ""
        if "<|reasoning|>" in r and "<|theory_ref|>" in r:
            txt = r[r.find("<|reasoning|>")+len("<|reasoning|>"):r.find("<|theory_ref|>")].strip()
        n = len(txt)
        if n > 500: s += 2.0
        elif n > 300: s += 1.5
        elif n > 150: s += 1.0
        elif n > 50:  s += 0.5
        s += min(sum(1 for kw in PHYSICS_KW if kw in txt.lower()) * 0.2, 2.0)
        out.append(s)
    return out

def decision_reward(responses, **kw):
    out = []
    for r in responses:
        if "<|decision|>" not in r:
            out.append(0.0); continue
        dec = r[r.find("<|decision|>")+len("<|decision|>"):].strip().lower()
        s = 1.0 + min(sum(1 for k in ACTION_KW if k in dec) * 0.5, 2.0)
        out.append(s)
    return out

def combined_reward(responses, **kw):
    f = format_reward(responses)
    r = reasoning_reward(responses)
    d = decision_reward(responses)
    return [a+b+c for a,b,c in zip(f,r,d)]

# Test
good = ["<|reasoning|>Dynamic pressure approaching max-Q. Reducing throttle protects structure. Delta-v loss minimal vs structural risk at this velocity and trajectory altitude.<|theory_ref|>Max-Q management<|decision|>Reduce throttle to 70%, maintain pitch"]
bad  = ["Go up."]
print(f"Good: {combined_reward(good)[0]:.1f}/10  Bad: {combined_reward(bad)[0]:.1f}/10")
print("✓ Reward functions ready")

## Step 4 — Prepare GRPO Dataset

In [ ]:
with open(TRAINING_DATA) as f:
    sft_data = [json.loads(l) for l in f if l.strip()]

grpo_prompts = []
for item in sft_data:
    msgs   = item["messages"][:2]   # system + user only
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    grpo_prompts.append({"prompt": prompt})

random.shuffle(grpo_prompts)
split      = int(len(grpo_prompts) * 0.95)
grpo_train = Dataset.from_list(grpo_prompts[:split])
grpo_val   = Dataset.from_list(grpo_prompts[split:])

print(f"✓ GRPO dataset — train: {len(grpo_train)}  val: {len(grpo_val)}")
print(f"Sample prompt:\n{grpo_train[0]['prompt'][:300]}")

## Step 5 — GRPO Training

Generates 8 responses per prompt, scores all 8, reinforces best.
No KSP. Pure reasoning quality improvement. Expected: 2-4 hours.

In [ ]:
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    os.environ["WANDB_PROJECT"] = "space-flight-ai-grpo"
    report_to = "wandb"
else:
    report_to = "none"

grpo_config = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    num_train_epochs=GRPO_EPOCHS,
    per_device_train_batch_size=GRPO_BATCH_SIZE,
    learning_rate=GRPO_LR,
    bf16=True, fp16=False,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to=report_to,
    num_generations=GRPO_GROUP_SIZE,
    max_new_tokens=600,
    temperature=0.9,
    seed=42,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=combined_reward,
    train_dataset=grpo_train,
    eval_dataset=grpo_val,
    processing_class=tokenizer,
)

print(f"Starting GRPO — {GRPO_GROUP_SIZE} responses per prompt, learns from best vs worst")
grpo_result = grpo_trainer.train()
grpo_trainer.model.save_pretrained(f"{GRPO_OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{GRPO_OUTPUT_DIR}/final")
print(f"✓ GRPO complete — loss: {grpo_result.metrics.get('train_loss', 'N/A')}")
print(f"  Saved: {GRPO_OUTPUT_DIR}/final")

## Step 6 — Test Reasoning Quality After GRPO

In [ ]:
class SafeLogitsProcessor(LogitsProcessor):
    def __call__(self, input_ids, scores):
        scores = torch.nan_to_num(scores, nan=-1e4, posinf=1e4, neginf=-1e4)
        return torch.clamp(scores, min=-1e4, max=1e4)

safe_proc = LogitsProcessorList([SafeLogitsProcessor()])

def ask_model(mdl, question, max_new_tokens=400):
    msgs   = [{"role":"system","content":"You are an AI rocket pilot. Analyse situations and provide detailed reasoning."},
               {"role":"user","content":question}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(next(mdl.parameters()).device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9, top_k=50,
            repetition_penalty=1.3, pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id, logits_processor=safe_proc)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

test_q = "<|mission_phase|>Gravity Turn\n<|situation|>Altitude 18km, velocity 420 m/s, dynamic pressure 28000 Pa, fuel 71%.\nProvide reasoning and response."
print("GRPO MODEL:")
print("="*60)
response = ask_model(grpo_trainer.model, test_q)
print(response)
print(f"\nReward: {combined_reward([response])[0]:.1f}/10")

---
# PHASE B — PPO Flight Control
### Requires KSP1 + kRPC + SSH tunnel.
---

## KSP + SSH Tunnel Setup

**Local Windows machine:**
1. Install KSP1 (Steam)
2. Install kRPC mod: https://krpc.github.io/krpc/
3. KSP → Settings → kRPC → Start Server (address: `0.0.0.0`, port: `50000`)
4. Place rocket on launchpad

**SSH tunnel (run on local machine):**
```
ssh -R 50000:localhost:50000 -R 50001:localhost:50001 jovyan@<H200-IP>
```

In [ ]:
# Test KSP connection before running PPO
try:
    import krpc
    conn   = krpc.connect(name="SpaceFlightAI_Test", address=KSP_HOST,
                          rpc_port=KSP_RPC_PORT, stream_port=KSP_STREAM_PORT)
    vessel = conn.space_center.active_vessel
    print(f"✓ KSP connected — vessel: {vessel.name}")
    print(f"  Altitude: {vessel.flight().mean_altitude:.0f}m")
    conn.close()
    KSP_AVAILABLE = True
except Exception as e:
    print(f"✗ KSP not available: {e}")
    KSP_AVAILABLE = False

## Step 7 — KSP Bridge

In [ ]:
import krpc

class KSPBridge:
    """Phase 1 action space: throttle + pitch + heading. ABORT never AI controlled."""

    def __init__(self):
        self.conn = self.vessel = None

    def connect(self):
        try:
            self.conn   = krpc.connect(name="SpaceFlightAI", address=KSP_HOST,
                                       rpc_port=KSP_RPC_PORT, stream_port=KSP_STREAM_PORT)
            self.vessel = self.conn.space_center.active_vessel
            print(f"✓ KSP — {self.vessel.name}"); return True
        except Exception as e:
            print(f"✗ {e}"); return False

    def get_telemetry(self):
        v = self.vessel; f = v.flight(); o = v.orbit
        r = v.resources
        lf_tot = r.max("LiquidFuel")
        lf_cur = r.amount("LiquidFuel")
        return {
            "altitude"         : round(f.mean_altitude, 1),
            "velocity"         : round(f.speed, 1),
            "vertical_velocity": round(f.vertical_speed, 1),
            "pitch"            : round(f.pitch, 1),
            "heading"          : round(f.heading, 1),
            "dynamic_pressure" : round(f.dynamic_pressure, 1),
            "apoapsis"         : round(o.apoapsis_altitude, 1),
            "periapsis"        : round(o.periapsis_altitude, 1),
            "eccentricity"     : round(o.eccentricity, 4),
            "fuel_remaining"   : round((lf_cur/lf_tot*100) if lf_tot>0 else 0, 1),
            "throttle"         : round(v.control.throttle, 2),
            "situation"        : str(v.situation),
        }

    def send_action(self, action):
        ctrl = self.vessel.control; ap = self.vessel.auto_pilot
        if action.get("throttle") is not None:
            ctrl.throttle = float(max(0.0, min(1.0, action["throttle"])))
        if action.get("pitch") is not None and action.get("heading") is not None:
            ap.engage()
            ap.target_pitch_and_heading(float(action["pitch"]), float(action["heading"]))
        if action.get("stage_suggested"):
            print("  [STAGING SUGGESTION] Press Space in KSP to confirm")

    def is_alive(self):
        try: _ = self.vessel.situation; return True
        except: return False

    def disconnect(self):
        if self.conn: self.conn.close()

print("✓ KSP Bridge defined")

## Step 8 — Reward Functions Per Level

In [ ]:
class MissionRewards:

    @staticmethod
    def level1_hover(t, target=500):
        r = 1.0 - abs(t["altitude"]-target)*0.01 - abs(t["vertical_velocity"])*0.1
        if t["altitude"] < 10: r -= 100
        if abs(t["altitude"]-target)<10 and abs(t["vertical_velocity"])<1: r += 10
        return r

    @staticmethod
    def level2_vtol(t):
        r = 0.5
        if t["altitude"] < 5:
            vv = abs(t["vertical_velocity"])
            r += 500 if vv<2 else (100 if vv<5 else -200)
        return r

    @staticmethod
    def level3_orbit(t, target_apo=80000):
        r = 0.1 - abs(t["apoapsis"]-target_apo)*0.001 - t["eccentricity"]*50
        if t["periapsis"] > 70000: r += 1000
        if t["altitude"] < 0: r -= 500
        return r

    @staticmethod
    def level4_hohmann(t, target=250000):
        r = -abs(t["apoapsis"]-target)*0.0005 - abs(t["periapsis"]-target)*0.0005 - t["eccentricity"]*100
        if abs(t["apoapsis"]-target)<5000 and abs(t["periapsis"]-target)<5000: r += 2000
        return r

    @staticmethod
    def level5_interplanetary(t):
        return 5000.0 if t["apoapsis"] > 84159286 else 0.0

    @staticmethod
    def level6_rendezvous(t, dist=None):
        if dist is None: return 0.0
        r = -dist*0.01 - t.get("relative_velocity",999)*0.5
        if dist < 100: r += 3000
        return r


CURRICULUM = {
    1: {"name":"Hover",          "fn":MissionRewards.level1_hover,         "threshold":50,   "phase":"Hover Control"},
    2: {"name":"VTOL",           "fn":MissionRewards.level2_vtol,          "threshold":100,  "phase":"Vertical Landing"},
    3: {"name":"Orbit",          "fn":MissionRewards.level3_orbit,         "threshold":500,  "phase":"Orbital Insertion"},
    4: {"name":"Hohmann",        "fn":MissionRewards.level4_hohmann,       "threshold":1000, "phase":"Hohmann Transfer"},
    5: {"name":"Interplanetary", "fn":MissionRewards.level5_interplanetary,"threshold":2000, "phase":"Interplanetary"},
    6: {"name":"Rendezvous",     "fn":MissionRewards.level6_rendezvous,    "threshold":1500, "phase":"Rendezvous"},
}

print("✓ Curriculum:")
for k,v in CURRICULUM.items():
    print(f"  L{k}: {v['name']:<20} threshold: {v['threshold']}")

## Step 9 — Telemetry Prompt Builder + Action Parser

In [ ]:
def telemetry_to_prompt(t, phase):
    return (
        f"<|mission_phase|>{phase}\n"
        f"<|telemetry|>altitude:{t['altitude']}m velocity:{t['velocity']}m/s "
        f"vv:{t['vertical_velocity']}m/s pitch:{t['pitch']}deg "
        f"dyn_pressure:{t['dynamic_pressure']}Pa apo:{t['apoapsis']}m "
        f"peri:{t['periapsis']}m fuel:{t['fuel_remaining']}%\n"
        f"<|situation|>Analyse flight state and decide next action. Provide reasoning and response."
    )

def parse_action(response):
    action = {"throttle":None,"pitch":None,"heading":None,"stage_suggested":False,"raw":""}
    if "<|decision|>" not in response: return action
    dec = response[response.find("<|decision|>")+len("<|decision|>"):].strip()
    action["raw"] = dec
    t = dec.lower()
    m = re.search(r'throttle[^0-9]*([0-9]+(?:\.[0-9]+)?)\s*%?', t)
    if m:
        v = float(m.group(1))
        action["throttle"] = v/100 if v>1 else v
    m = re.search(r'pitch[^0-9-]*(-?[0-9]+(?:\.[0-9]+)?)', t)
    if m: action["pitch"] = float(m.group(1))
    m = re.search(r'heading[^0-9]*([0-9]+)', t)
    if m: action["heading"] = float(m.group(1))
    if "east"  in t: action["heading"] = 90.0
    if "west"  in t: action["heading"] = 270.0
    if "north" in t: action["heading"] = 0.0
    if "south" in t: action["heading"] = 180.0
    if any(w in t for w in ["stage","separate","jettison"]): action["stage_suggested"] = True
    return action

# Quick test
test = "<|reasoning|>...<|theory_ref|>...<|decision|>throttle 80%, pitch 45, heading east"
p = parse_action(test)
print(f"Parser: throttle={p['throttle']} pitch={p['pitch']} heading={p['heading']}")
print("✓ Action parser ready")

## Step 10 — Flight Logger (Every Log Saved Automatically)

In [ ]:
class FlightLogger:
    def __init__(self, level, episode):
        self.level = level; self.episode = episode
        self.entries = []; self.t0 = time.time()
        self.filename = f"{FLIGHT_LOG_DIR}/L{level}_ep{episode:04d}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

    def log(self, telemetry, response, action, reward):
        self.entries.append({
            "mission_time"      : round(time.time()-self.t0, 1),
            "telemetry"         : telemetry,
            "llm_response"      : response,
            "action_taken"      : action,
            "reward"            : round(reward, 3),
            "engineer_feedback" : None   # engineer fills this in
        })

    def save(self):
        rewards = [e["reward"] for e in self.entries]
        with open(self.filename, "w") as f:
            json.dump({
                "level":CURRICULUM[self.level]["name"],
                "episode":self.episode,
                "duration_sec":round(time.time()-self.t0,1),
                "total_reward":round(sum(rewards),2),
                "total_steps":len(rewards),
                "entries":self.entries
            }, f, indent=2)
        return self.filename

    def summary(self):
        r = [e["reward"] for e in self.entries]
        return {"total_reward":round(sum(r),2),"avg":round(sum(r)/len(r) if r else 0,3),"steps":len(r)}

print(f"✓ Flight logger — saves every log to {FLIGHT_LOG_DIR}/")

## Step 11 — Curriculum Manager

In [ ]:
class CurriculumManager:
    """
    Advances when BOTH:
    1. >= 3 successes at current level
    2. avg reward (last 10 eps) > 80% of threshold
    """
    def __init__(self):
        self.current = 1
        self.successes = {l:0 for l in CURRICULUM}
        self.recent    = {l:deque(maxlen=10) for l in CURRICULUM}

    def record(self, total_reward):
        lvl = self.current
        self.recent[lvl].append(total_reward)
        if total_reward >= CURRICULUM[lvl]["threshold"]: self.successes[lvl] += 1
        return self._check_advance()

    def _check_advance(self):
        lvl = self.current; cfg = CURRICULUM[lvl]
        recent = list(self.recent[lvl])
        if len(recent) < 3: return False
        ok_successes = self.successes[lvl] >= SUCCESSES_TO_ADVANCE
        ok_avg       = sum(recent)/len(recent) > cfg["threshold"] * REWARD_THRESHOLD_PCT
        if ok_successes and ok_avg:
            if lvl < max(CURRICULUM):
                self.current += 1
                print(f"\n{'='*50}")
                print(f"ADVANCING TO LEVEL {self.current}: {CURRICULUM[self.current]['name']}")
                print(f"{'='*50}\n")
                return True
            else:
                print("\n🚀 ALL LEVELS COMPLETE")
        return False

    def status(self):
        lvl = self.current; r = list(self.recent[lvl])
        avg = sum(r)/len(r) if r else 0
        print(f"L{lvl} {CURRICULUM[lvl]['name']} | successes:{self.successes[lvl]}/{SUCCESSES_TO_ADVANCE} | avg:{avg:.1f}")

print(f"✓ Curriculum manager — advances after {SUCCESSES_TO_ADVANCE} successes + avg>{REWARD_THRESHOLD_PCT*100:.0f}% threshold")

## Step 12 — Run PPO Curriculum Training

In [ ]:
def run_episode(mdl, ksp, level, episode, max_steps=300):
    cfg = CURRICULUM[level]; logger = FlightLogger(level, episode); total = 0.0
    print(f"  Ep {episode} L{level}: {cfg['name']}")
    for step in range(max_steps):
        if not ksp.is_alive(): print(f"  Vessel lost step {step}"); break
        t        = ksp.get_telemetry()
        prompt   = telemetry_to_prompt(t, cfg["phase"])
        response = ask_model(mdl, prompt, max_new_tokens=300)
        action   = parse_action(response)
        ksp.send_action(action)
        time.sleep(0.5)
        t2     = ksp.get_telemetry()
        reward = cfg["fn"](t2)
        total += reward
        logger.log(t, response, action, reward)
        if step % 25 == 0:
            print(f"  step{step:3d} alt:{t['altitude']:8.0f}m r:{reward:6.1f} total:{total:8.1f}")
        if total >= cfg["threshold"]: print(f"  ✓ L{level} SUCCESS step {step}"); break
    log_file = logger.save()
    print(f"  Log: {log_file}")
    return logger


if not KSP_AVAILABLE:
    print("KSP not available — set up SSH tunnel first")
else:
    ppo_model  = grpo_trainer.model if 'grpo_trainer' in dir() else model
    ksp        = KSPBridge()
    curriculum = CurriculumManager()

    if ksp.connect():
        if WANDB_KEY:
            wandb.init(project="space-flight-ai-ppo")

        for episode in range(1, 10001):
            curriculum.status()
            logger   = run_episode(ppo_model, ksp, curriculum.current, episode)
            summary  = logger.summary()
            advanced = curriculum.record(summary["total_reward"])

            if advanced:
                ckpt = f"{PPO_OUTPUT_DIR}/level_{curriculum.current-1}_complete"
                ppo_model.save_pretrained(ckpt)
                print(f"  Checkpoint: {ckpt}")

            if WANDB_KEY and wandb.run:
                wandb.log({"episode":episode,"level":curriculum.current,
                           "total_reward":summary["total_reward"],"avg":summary["avg"]})

        ksp.disconnect()

## Step 13 — Engineer Feedback Pipeline

**Use after Level 3 (orbit) is working.**
Engineer opens flight log JSON, fills `engineer_feedback` fields, saves.
Run this cell to convert corrections into new training pairs.

In [ ]:
def process_feedback(log_file):
    with open(log_file) as f: log = json.load(f)
    lvl   = next(k for k,v in CURRICULUM.items() if v["name"]==log["level"])
    phase = CURRICULUM[lvl]["phase"]
    pairs = []
    for entry in log["entries"]:
        if not entry.get("engineer_feedback"): continue
        pairs.append({"messages":[
            {"role":"system","content":"You are an AI rocket pilot. Analyse the situation and provide detailed reasoning."},
            {"role":"user",  "content":telemetry_to_prompt(entry["telemetry"], phase)},
            {"role":"assistant","content":f"<|correction|>{entry['engineer_feedback']}"}
        ], "source":"engineer_feedback"})
    print(f"Extracted {len(pairs)} feedback pairs")
    return pairs

def save_feedback(pairs, out="./engineer_feedback.jsonl"):
    with open(out, "a") as f:
        for p in pairs: f.write(json.dumps(p)+"\n")
    print(f"✓ {len(pairs)} pairs → {out}")
    print("  Merge with finetune_ready.jsonl and rerun notebook 02")

# List available logs
logs = sorted(Path(FLIGHT_LOG_DIR).glob("*.json"))
print(f"Flight logs ready for review: {len(logs)}")
for log in logs[-5:]:
    with open(log) as f: d = json.load(f)
    print(f"  {log.name}  reward:{d['total_reward']}  steps:{d['total_steps']}")

# Usage:
# pairs = process_feedback("./flight_logs/L3_ep0023_....json")
# save_feedback(pairs)

## Step 14 — Save Final RL Model

In [ ]:
final = grpo_trainer.model if 'grpo_trainer' in dir() else model
final.save_pretrained(FINAL_RL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_RL_MODEL_DIR)
with open(f"{FINAL_RL_MODEL_DIR}/rl_summary.json","w") as f:
    json.dump({"base":BASE_MODEL,"sft":SFT_MODEL_DIR,
               "grpo_epochs":GRPO_EPOCHS,"timestamp":datetime.now().isoformat()}, f, indent=2)
print(f"✓ Final RL model → {FINAL_RL_MODEL_DIR}")
print(f"  Upload: HfApi().upload_folder('{FINAL_RL_MODEL_DIR}', 'Pegasus167/space-flight-ai-rl', repo_type='model')")

---
## Summary

| Phase | What | Output | Engineer needed? |
|---|---|---|---|
| GRPO | Reasoning quality | `grpo_checkpoints/final/` | No |
| PPO L1-L2 | Hover + VTOL | level checkpoints | No |
| PPO L3+ | Orbit onwards | level checkpoints | Yes — after L3 works |
| Feedback | Corrections → SFT pairs | `engineer_feedback.jsonl` | Yes |

**Next:** `04_evaluation.ipynb`